# v6: the matcher learns from 100% of train entities

Needs on Drive: `ber_v3/parquet/`, `ber_v3/work_ids_ce10.parquet`, `ber_v3/ckpt/`. Run cells 1-4, then cell 5 (~1.3 h).
Before v5 has finished this gives v4 + all train data (`ber_v3/out_v6/`); after v5 has saved its LLM scores to the
checkpoint it gives v5 + all train data (`ber_v3/out_v6_llm/`). Log: `ber_v3/run_v6.log`.


In [1]:
!nproc; free -g | head -2; nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/ber_v3'
CE_DRIVE = '/content/drive/MyDrive/ber_ce/ce_model'
!ls -la {DRIVE} {DRIVE}/parquet {CE_DRIVE}

Mounted at /content/drive
/content/drive/MyDrive/ber_ce/ce_model:
total 1102843
drwx------ 2 root root       4096 Sep 26 12:41 .
drwx------ 3 root root       4096 Sep 26 11:42 ..
-rw------- 1 root root        840 Sep 26 12:41 config.json
-rw------- 1 root root        198 Sep 26 12:41 metrics.json
-rw------- 1 root root 1112201932 Sep 26 12:41 model.safetensors
-rw------- 1 root root        343 Sep 26 12:41 tokenizer_config.json
-rw------- 1 root root   17098086 Sep 26 12:41 tokenizer.json

/content/drive/MyDrive/ber_v3:
total 1670
drwx------ 8 root root    4096 Sep 26 14:12 .
drwx------ 6 root root    4096 Sep 27 09:18 ..
drwx------ 2 root root    4096 Sep 27 00:30 ckpt
-rw------- 1 root root     119 Sep 27 08:33 features.log
drwx------ 2 root root    4096 Sep 26 22:42 out
drwx------ 2 root root    4096 Sep 26 22:42 out_v3_models
drwx------ 2 root root    4096 Sep 27 04:10 out_v4
drwx------ 2 root root    4096 Sep 27 08:56 out_v5
drwx------ 2 root root    4096 Sep 26 14:12 parquet
-rw-

In [2]:
CODE = '...'

In [3]:
# Unpack the pipeline code embedded in the cell above (rebuilt from the repo by colab/build_notebook.py).
import base64, io, zipfile, shutil, os
shutil.rmtree('/content/code', ignore_errors=True)
zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE))).extractall('/content/code')
print(sorted(os.listdir('/content/code/src')))

['__init__.py', 'bench_translit.py', 'candidates.py', 'ce_data.py', 'config.py', 'context.py', 'cross_encoder.py', 'data.py', 'decode.py', 'eda.py', 'eval_retrieval.py', 'features.py', 'filter_sweep.py', 'filtering.py', 'lgbutil.py', 'llm_reranker.py', 'metric.py', 'normalize.py', 'pipeline_v3.py', 'pipeline_v4.py', 'predict.py', 'prepare.py', 'redecode.py', 'retrieve.py', 'submission.py', 'text.py', 'train.py', 'train2.py', 'translit.py']


In [4]:
# Setup: data and a separate environment with the pinned packages.
import os, sys
!mkdir -p /content/work/parquet /content/work/cand /content/work/output
!cp -n {DRIVE}/parquet/*.parquet /content/work/parquet/ && cp -n {DRIVE}/work_ids_ce10.parquet /content/work/cand/ && ls -la /content/work/parquet
!pip install -q uv && uv venv -q --allow-existing /content/venv && uv pip install -q --python /content/venv/bin/python -r /content/code/requirements.txt
os.environ.update({'BER_WORK_DIR': '/content/work', 'BER_OUTPUT_DIR': '/content/work/output', 'BER_THREADS': '46',
                   'BER_CE_PYTHON': sys.executable, 'PYTHONIOENCODING': 'utf-8', 'BER_K': '20',
                   'BER_CAND_BUDGET': '16', 'BER_MODEL_FRAC': '0.3'})
PY = '/content/venv/bin/python'
!{PY} -c "import pandas, lightgbm, sparse_dot_topn, rapidfuzz; print('pipeline env ok', pandas.__version__, lightgbm.__version__)"

cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
total 1642644
drwxr-xr-x 2 root root      4096 Sep 27 09:19 .
drwxr-xr-x 5 root root      4096 Sep 27 09:19 ..
-rw------- 1 root root 110464226 Sep 27 09:19 test_source1.parquet
-rw------- 1 root root 336017967 Sep 27 09:19 test_source2.parquet
-rw------- 1 root root 335537737 Sep 27 09:19 test_source3.parquet
-rw------- 1 root root  92419561 Sep 27 09:19 train_ground_truth.parquet
-rw------- 1 root root 136677134 Sep 27 09:19 train_source1.parquet
-rw------- 1 root root 332018842 Sep 27 09:19 train_source2.parquet
-rw------- 1 root root 338894978 Sep 27 09:19 train_source3.parquet
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 178.0 MB/s eta 0:00:0000:0100:01
pipeline env ok 3.0.6 4.7.0


In [5]:
# v6, unattended (~1.3 h): restore the checkpoint (kept pairs, cross-encoder scores, and the LLM scores once v5 has
# saved them), normalize, recompute features, then stages 1-2 on 100% of train entities and predict. Results go to
# ber_v3/out_v6 (or out_v6_llm when the LLM scores were there); log in ber_v3/run_v6.log. Releases the runtime.
import os, sys, time, subprocess, glob, shutil
os.environ['BER_CE_PYTHON'] = sys.executable
# Colab ships torchao 0.10, which current peft refuses to import alongside; nothing here needs torchao.
!pip uninstall -y -q torchao
!pip install -q peft
!{sys.executable} -c "import peft; print('peft', peft.__version__)"
LOG = '/content/work/run.log'

def run(module, steps):
    ok = subprocess.run(f'cd /content/code && {PY} -m src.{module} {steps} >> {LOG} 2>&1', shell=True).returncode == 0
    print(time.strftime('%H:%M:%S'), 'OK' if ok else 'FAILED', module, steps, flush=True)
    shutil.copy(LOG, DRIVE)
    return ok

def to_drive(patterns, dest):
    try:
        for pattern in patterns:
            for src in glob.glob(pattern):
                if os.path.isfile(src):
                    dst = os.path.join(DRIVE, dest, os.path.relpath(src, '/content/work'))
                    os.makedirs(os.path.dirname(dst), exist_ok=True)
                    shutil.copy(src, dst)
        print(time.strftime('%H:%M:%S'), 'saved to Drive:', dest, flush=True)
    except OSError as e:
        print('Drive copy failed, continuing:', e, flush=True)

def outputs(name):
    to_drive(['/content/work/output/*', '/content/work/v3/validation.json', '/content/work/v3/val_pairs.parquet',
              '/content/work/v3/filter_*'], name)

LOG = '/content/work/run_v6.log'
!mkdir -p /content/work/v3 && cp -r {DRIVE}/ckpt/v3/. /content/work/v3/ && ls /content/work/v3/ce
os.environ['BER_STAGE_ALL'] = '1'
name = 'out_v6_llm' if len(glob.glob('/content/work/v3/ce/*_llm_scores.parquet')) == 2 else 'out_v6'
print('results will go to', name, flush=True)
if run('pipeline_v3', 'prepare features') and run('pipeline_v4', 'stage1 stage2 predict'):
    outputs(name)
    to_drive(['/content/work/v3/p1_test.npy'], name)
print(time.strftime('%H:%M:%S'), 'finished; releasing the runtime', flush=True)
from google.colab import runtime
runtime.unassign()

peft 0.20.0
records_test.parquet	      train_large_m0_scores.parquet
records_train.parquet	      train_large_m1_scores.parquet
test_large_m0_scores.parquet  train_large_scores.parquet
test_large_scores.parquet     train_llm_scores.parquet
test_llm_scores.parquet       train_m0_scores.parquet
test_m0_scores.parquet	      train_m1_scores.parquet
test_pairs.parquet	      train_pairs.parquet
test_scores.parquet	      train_scores.parquet
results will go to out_v6_llm
09:35:01 OK pipeline_v3 prepare features
10:21:11 OK pipeline_v4 stage1 stage2 predict
10:21:14 saved to Drive: out_v6_llm
10:21:14 saved to Drive: out_v6_llm
10:21:14 finished; releasing the runtime
